# 5G Network Automation Using Local LLMs and RAG

[![arXiv](https://img.shields.io/badge/arXiv-2511.21084-b31b1b.svg)](https://arxiv.org/abs/2511.21084)
[![6G Summit](https://img.shields.io/badge/6G%20Summit-Abu%20Dhabi%202024-blue)](https://eucnc.eu/)

**Authors:** Ahmadreza Majlesara, Ali Majlesi, Ali Mamaghani, Alireza Shokrani, Babak H. Khalaj  
**Affiliation:** Sharif University of Technology  
**Venue:** 3rd 6G Summit, Abu Dhabi, UAE — November 2024

---

### Pipeline Overview

This notebook implements a **two-stage RAG pipeline** for 5G network command automation:

| Stage | Task | Method |
|-------|------|--------|
| 1 | **Classification** | Classify NL query into one of 12 command categories using RAG few-shot |
| 2 | **Generation** | Generate the exact CLI command using RAG-augmented prompting |

**Key results (test set):**
- Classification accuracy: **94.29%** (RAG Classifier Retriever)
- End-to-end exact match: ~45–46%
- Mean IoU: 65–68%

### Table of Contents
1. [Dependencies](#1.-Dependencies)
2. [Imports](#2.-Imports)
3. [Dataset](#3.-Dataset)
4. [LLM Setup (Ollama)](#4.-LLM-Setup)
5. [Vector Store — Embedding Extraction](#5.-Vector-Store)
6. [Retriever Setup](#6.-Retriever-Setup)
7. [Stage 1: Command Classification](#7.-Stage-1:-Classification)
8. [Stage 2: Command Generation](#8.-Stage-2:-Generation)
9. [Evaluation — Classification Accuracy](#9.-Evaluation:-Classification)
10. [Evaluation — Generation Accuracy & Mean IoU](#10.-Evaluation:-Generation)

## 1. Dependencies

In [ ]:
%%capture
!pip install chromadb
!pip install llama-index
!pip install llama-index-vector-stores-chroma
!pip install llama-index-embeddings-huggingface

## 2. Imports

In [ ]:
import os
import re
import random
import shutil
import zipfile
from tqdm import tqdm

import torch
import gdown
import ollama

import chromadb
from llama_index.core import (
    VectorStoreIndex,
    StorageContext,
    Settings,
    SimpleDirectoryReader,
    Document,
)
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print(f"torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

## 3. Dataset

The dataset contains natural language descriptions of 5G network operations paired with their ground-truth CLI commands.  
It is organized by command category, each with `in/` (descriptions) and `out/` (commands) subdirectories.

> **Skip this cell** if `LLM_app_dataset_v2.zip` is already in the working directory.

In [ ]:
# Download dataset from Google Drive
gdown.download(id='1BROb8jjYJc-P1voevE9OLcptuF7UMLK7', output='./LLM_app_dataset_v2.zip')

os.makedirs('./LLM_app_dataset', exist_ok=True)
with zipfile.ZipFile('./LLM_app_dataset_v2.zip', 'r') as zf:
    zf.extractall('./LLM_app_dataset')

print('Dataset extracted.')

### 3.1 Build Test Splits

We randomly sample **25% of each category** for evaluation, keeping the rest for the RAG vector store.

In [ ]:
def build_test_split(base_dir: str, dest_name: str, skip_folder: str = None):
    """
    Sample 25% of each category's examples into a held-out test split.

    Args:
        base_dir    : Root directory containing category subfolders.
        dest_name   : Name of the output test-split directory.
        skip_folder : Optional category folder to exclude (e.g. 'classifier_dataset').
    """
    dest_in  = os.path.join(dest_name, 'in')
    dest_out = os.path.join(dest_name, 'out')
    os.makedirs(dest_in,  exist_ok=True)
    os.makedirs(dest_out, exist_ok=True)

    counter = 1
    for folder in os.listdir(base_dir):
        if folder == skip_folder:
            continue
        folder_path = os.path.join(base_dir, folder)
        in_path  = os.path.join(folder_path, 'in')
        out_path = os.path.join(folder_path, 'out')

        if not (os.path.isdir(folder_path) and os.path.exists(in_path)):
            continue

        in_files = os.listdir(in_path)
        sampled  = random.sample(in_files, max(1, len(in_files) // 4))

        for fname in sampled:
            num   = re.search(r'\d+', fname)
            if not num:
                continue
            n     = num.group()
            out_f = f'out{n}.txt'

            src_in  = os.path.join(in_path,  fname)
            src_out = os.path.join(out_path, out_f)

            if os.path.exists(src_in) and os.path.exists(src_out):
                shutil.copy(src_in,  os.path.join(dest_in,  f'in{counter}.txt'))
                shutil.copy(src_out, os.path.join(dest_out, f'out{counter}.txt'))
                counter += 1

    print(f'Built {dest_name}: {counter - 1} examples')


# Generation test split (all categories except classifier)
build_test_split('.', 'test_dataset', skip_folder='classifier_dataset')

# Classification test split (classifier category only)
build_test_split('.', 'classifier_test_dataset', skip_folder=None)

## 4. LLM Setup

We use **LLaMA-3 8B Instruct (Q4 quantized)** running locally via [Ollama](https://ollama.com).  
Make sure Ollama is running: `ollama serve`

In [ ]:
MODEL_NAME = 'llama3'  # Ollama model identifier

def llama3(prompt: str, system_text: str) -> str:
    """
    Query LLaMA-3 via Ollama.

    Args:
        prompt      : User message.
        system_text : System prompt defining the assistant's role.

    Returns:
        Model response as a string.
    """
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {'role': 'system', 'content': system_text},
            {'role': 'user',   'content': prompt},
        ]
    )
    return response['message']['content']


# Sanity check
resp = llama3('What is Open5GS?', 'You are a concise telecom assistant.')
print(resp[:300])

## 5. Vector Store

We build two persistent ChromaDB indexes using **BAAI/bge-small-en-v1.5** embeddings:
- `rag_vector` — general command corpus (used by the generator)
- `classifier_rag_vector` — classifier-specific corpus (used by Stage 1)

In [ ]:
# Embedding model configuration
Settings.embed_model = HuggingFaceEmbedding(model_name='BAAI/bge-small-en-v1.5')
Settings.llm         = None   # We handle LLM calls manually
Settings.chunk_size  = 128
Settings.chunk_overlap = 20

# Control flags — set to False to reuse an existing index
BUILD_RAG_INDEX        = True  # @param {type:"boolean"}
BUILD_CLASSIFIER_INDEX = True  # @param {type:"boolean"}

In [ ]:
def build_chroma_index(
    data_root: str,
    vector_path: str,
    collection_name: str,
    skip_folder: str = None,
    only_folder: str = None,
) -> VectorStoreIndex:
    """
    Build a ChromaDB vector index from a directory of text files.

    Args:
        data_root       : Root directory to crawl for `in/` subfolders.
        vector_path     : Path to persist the ChromaDB database.
        collection_name : ChromaDB collection name.
        skip_folder     : Folder name to exclude from indexing.
        only_folder     : If set, index only this folder.

    Returns:
        A LlamaIndex VectorStoreIndex backed by ChromaDB.
    """
    os.makedirs(vector_path, exist_ok=True)

    all_documents = []
    for root, dirs, _ in os.walk(data_root):
        # Exclude test splits from the index
        for excluded in ['test_dataset', 'classifier_test_dataset']:
            if excluded in dirs:
                dirs.remove(excluded)

        if only_folder and os.path.basename(os.path.dirname(root)) != only_folder:
            if os.path.basename(root) == 'in' and only_folder not in root:
                continue

        if skip_folder and skip_folder in root:
            continue

        if os.path.basename(root) == 'in':
            docs = SimpleDirectoryReader(root).load_data()
            all_documents.extend(docs)

    db               = chromadb.PersistentClient(path=vector_path)
    collection       = db.get_or_create_collection(collection_name)
    vector_store     = ChromaVectorStore(chroma_collection=collection)
    storage_context  = StorageContext.from_defaults(vector_store=vector_store)

    index = VectorStoreIndex.from_documents(
        all_documents,
        storage_context=storage_context,
        show_progress=True,
    )

    print(f'Index built: {len(all_documents)} documents → {vector_path}')
    return index


if BUILD_RAG_INDEX:
    rag_index = build_chroma_index(
        data_root='.',
        vector_path='./rag_vector',
        collection_name='index_cleaned',
        skip_folder='classifier_dataset',
    )

if BUILD_CLASSIFIER_INDEX:
    classifier_index = build_chroma_index(
        data_root='.',
        vector_path='./classifier_rag_vector',
        collection_name='index_cleaned',
        only_folder='classifier_dataset',
    )

## 6. Retriever Setup

Load the persisted indexes and configure retrievers with **top-k = 8**.

In [ ]:
TOP_K = 8

def load_index(vector_path: str, collection_name: str) -> VectorStoreIndex:
    """Load a persisted ChromaDB-backed index."""
    db           = chromadb.PersistentClient(path=vector_path)
    collection   = db.get_or_create_collection(collection_name)
    vector_store = ChromaVectorStore(chroma_collection=collection)
    storage_ctx  = StorageContext.from_defaults(vector_store=vector_store)
    return VectorStoreIndex.from_vector_store(vector_store, storage_context=storage_ctx)


# Load indexes
rag_index        = load_index('./rag_vector',        'index_cleaned')
classifier_index = load_index('./classifier_rag_vector', 'index_cleaned')

# Build retrievers and query engines
rag_retriever        = VectorIndexRetriever(index=rag_index,        similarity_top_k=TOP_K)
classifier_retriever = VectorIndexRetriever(index=classifier_index, similarity_top_k=TOP_K)

rag_engine        = RetrieverQueryEngine(retriever=rag_retriever)
classifier_engine = RetrieverQueryEngine(retriever=classifier_retriever)

print(f'Retrievers ready (top-k = {TOP_K})')

## 7. Stage 1: Classification

Given a natural language query, retrieve the most similar examples from the classifier corpus and ask LLaMA-3 to predict the command category in one word.

In [ ]:
VALID_CATEGORIES = [
    'classifier', 'completion', 'diag', 'extract', 'help',
    'list', 'login', 'network', 'observe', 'remove', 'test', 'user'
]


def read_file(path: str) -> str:
    """Read a text file, returning empty string on error."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read().strip()
    except Exception:
        return ''


def build_retrieval_dict(query: str, retrieval_result, base_dir: str = './') -> dict:
    """
    Build an {input_description: output_command} dict from retrieved nodes.
    Used to construct few-shot examples for the prompt.
    """
    result = {}
    for node in retrieval_result.source_nodes:
        node_text = node.text
        if node_text == query:
            continue
        # Find matching output file
        for root, _, files in os.walk(base_dir):
            if os.path.basename(root) != 'in':
                continue
            for fname in files:
                fpath = os.path.join(root, fname)
                if read_file(fpath) == node_text:
                    out_root = root.replace('/in', '/out').replace(os.sep + 'in', os.sep + 'out')
                    num      = re.search(r'\d+', fname)
                    if num:
                        out_path = os.path.join(out_root, f'out{num.group()}.txt')
                        if os.path.exists(out_path):
                            result[node_text] = read_file(out_path)
    return result


def build_few_shot_prompt(query: str, examples: dict) -> str:
    """Format retrieved examples as a few-shot prompt."""
    prompt = f'The input you have to give me output for is: {query}\n'
    prompt += 'Here are some similar inputs and outputs retrieved from the dataset:\n'
    for inp, out in examples.items():
        prompt += f'input: {inp} , output: {out}\n'
    prompt += f'Give me the output for this input: {query}.'
    return prompt


def classify_command(query: str) -> tuple[str, str]:
    """
    Stage 1: Classify a natural language query into a command category.

    Returns:
        (predicted_category, system_text_for_generation)
    """
    # Retrieve similar examples from classifier corpus
    retrieval      = classifier_engine.query(query)
    examples       = build_retrieval_dict(query, retrieval)
    few_shot_prompt = build_few_shot_prompt(query, examples)

    # Load classifier system prompt
    system_classifier = read_file('systemText/system_classifier.txt')

    # Ask LLaMA-3 for a one-word category
    classification_prompt = few_shot_prompt + '\ngive me the output in 1 word only.'
    raw_output = llama3(classification_prompt, system_classifier)

    # Match to valid category
    predicted = raw_output.strip().lower()
    for cat in VALID_CATEGORIES:
        if cat in predicted:
            predicted = cat
            break

    # Fallback: use category of first retrieved example
    if predicted not in VALID_CATEGORIES and examples:
        first_out = next(iter(examples.values()))
        for cat in VALID_CATEGORIES:
            if cat in first_out.lower():
                predicted = cat
                break

    # Load the corresponding system prompt for generation
    system_text = read_file(f'systemText/{predicted}.txt')

    return predicted, system_text


# Quick test
query = 'Is it possible for you to examine the network graph for any bugs and make necessary corrections?'
category, system_text = classify_command(query)
print(f'Query    : {query[:60]}...')
print(f'Category : {category}')

## 8. Stage 2: Generation

Using the predicted category (and its system prompt), retrieve relevant examples from the general RAG corpus and generate the CLI command.

In [ ]:
def generate_command(query: str, system_text: str) -> str:
    """
    Stage 2: Generate a 5G CLI command from a natural language description.

    Args:
        query       : Natural language description of the network operation.
        system_text : Category-specific system prompt (from Stage 1).

    Returns:
        Generated CLI command string.
    """
    # Retrieve relevant command examples from general corpus
    retrieval  = rag_engine.query(query)
    examples   = build_retrieval_dict(query, retrieval)
    prompt     = build_few_shot_prompt(query, examples)

    raw_output = llama3(prompt, system_text)

    # Extract the CLI command (pattern: 'br-t9s.cli ...')
    match = re.search(r'br-t9s\.cli[^\n]*', raw_output)
    if match:
        return match.group(0).strip('\'"')

    return raw_output.strip()


def run_pipeline(query: str, verbose: bool = True) -> dict:
    """Full two-stage pipeline: natural language → 5G CLI command."""
    category, system_text = classify_command(query)
    command               = generate_command(query, system_text)

    if verbose:
        print(f'Input    : {query}')
        print(f'Category : {category}')
        print(f'Command  : {command}')

    return {'query': query, 'category': category, 'command': command}


# Demo
print('=' * 60)
run_pipeline('Is it possible for you to examine the network graph for any bugs?')

## 9. Evaluation: Classification

We compare three approaches on the held-out `classifier_test_dataset`:
- **Method 1**: RAG with classifier-specific retriever
- **Method 2**: RAG with base retriever  
- **Method 3**: Zero-shot (no RAG examples)

In [ ]:
def calculate_accuracy(out_dir: str, pred_dir: str) -> float:
    """
    Compute exact-match accuracy between ground-truth and predicted outputs.

    Args:
        out_dir  : Directory of ground-truth output files.
        pred_dir : Directory of predicted output files.

    Returns:
        Accuracy as a float in [0, 1].
    """
    gt_files = [f for f in os.listdir(out_dir) if f.startswith('out') and f.endswith('.txt')]
    if not gt_files:
        return 0.0

    correct = 0
    for gt_file in gt_files:
        num      = gt_file.replace('out', '').replace('.txt', '')
        pred_file = os.path.join(pred_dir, f'response_in{num}.txt')
        gt_path   = os.path.join(out_dir, gt_file)

        if not os.path.exists(pred_file):
            continue

        gt_text   = read_file(gt_path).lower()
        pred_text = read_file(pred_file).lower()

        if gt_text in pred_text or pred_text in gt_text:
            correct += 1

    return correct / len(gt_files)

In [ ]:
base_dir    = 'classifier_test_dataset'
in_dir      = os.path.join(base_dir, 'in')
out_dir     = os.path.join(base_dir, 'out')
system_text = read_file(os.path.join(base_dir, 'system_classifier.txt'))

for method_name, retriever_engine in [
    ('RAG_ClassifierRetriever', classifier_engine),
    ('RAG_BaseRetriever',       rag_engine),
]:
    pred_dir = os.path.join(base_dir, method_name)
    os.makedirs(pred_dir, exist_ok=True)

    for fname in tqdm(os.listdir(in_dir), desc=method_name):
        if not fname.endswith('.txt'):
            continue
        query    = read_file(os.path.join(in_dir, fname))
        retrieval = retriever_engine.query(query)
        examples  = build_retrieval_dict(query, retrieval)
        prompt    = build_few_shot_prompt(query, examples) + '\ngive me the output in 1 word only.'
        prediction = llama3(prompt, system_text)

        num = re.search(r'\d+', fname)
        if num:
            with open(os.path.join(pred_dir, f'response_in{num.group()}.txt'), 'w') as f:
                f.write(prediction)

    acc = calculate_accuracy(out_dir, pred_dir)
    print(f'{method_name}: {acc:.2%}')

In [ ]:
# Zero-shot baseline (no RAG)
zero_shot_dir = os.path.join(base_dir, 'ZeroShot')
os.makedirs(zero_shot_dir, exist_ok=True)

for fname in tqdm(os.listdir(in_dir), desc='ZeroShot'):
    if not fname.endswith('.txt'):
        continue
    query      = read_file(os.path.join(in_dir, fname))
    prompt     = query + '\ngive me an only one word as the classification output.'
    prediction = llama3(prompt, system_text)

    num = re.search(r'\d+', fname)
    if num:
        with open(os.path.join(zero_shot_dir, f'response_in{num.group()}.txt'), 'w') as f:
            f.write(prediction)

acc_zs = calculate_accuracy(out_dir, zero_shot_dir)
print(f'ZeroShot: {acc_zs:.2%}')

In [ ]:
# Results table
print('\nClassification Results (Table 1 from paper)')
print('-' * 45)
print(f'{"Method":<30} {"Accuracy"}')
print('-' * 45)
print(f'{"RAG — Classifier Retriever":<30} 94.29%   ← paper')
print(f'{"RAG — Base Retriever":<30} 91.43%   ← paper')
print(f'{"Zero-Shot (no RAG)":<30} 74.29%   ← paper')
print('-' * 45)

## 10. Evaluation: Generation

Evaluate the full two-stage pipeline on the generation test set using:
- **Exact match accuracy** — predicted command == ground truth
- **Mean IoU** — token-level intersection over union (partial credit)

In [ ]:
def token_iou(pred: str, gt: str) -> float:
    """Token-level Intersection over Union between two command strings."""
    pred_tokens = set(pred.lower().split())
    gt_tokens   = set(gt.lower().split())
    if not pred_tokens and not gt_tokens:
        return 1.0
    if not pred_tokens or not gt_tokens:
        return 0.0
    return len(pred_tokens & gt_tokens) / len(pred_tokens | gt_tokens)


def calculate_miou(out_dir: str, pred_dir: str) -> float:
    """Compute mean IoU over all test examples."""
    gt_files = [f for f in os.listdir(out_dir) if f.startswith('out') and f.endswith('.txt')]
    ious     = []

    for gt_file in gt_files:
        num       = gt_file.replace('out', '').replace('.txt', '')
        pred_file = os.path.join(pred_dir, f'response_in{num}.txt')
        if not os.path.exists(pred_file):
            continue
        gt_text   = read_file(os.path.join(out_dir, gt_file))
        pred_text = read_file(pred_file)
        ious.append(token_iou(pred_text, gt_text))

    return sum(ious) / len(ious) if ious else 0.0

In [ ]:
gen_base   = 'test_dataset'
gen_in_dir = os.path.join(gen_base, 'in')
gen_out_dir= os.path.join(gen_base, 'out')

for method_name, use_classifier_retriever in [
    ('Pipeline_ClassifierRetriever', True),
    ('Pipeline_BaseRetriever',       False),
]:
    pred_dir = os.path.join(gen_base, method_name)
    os.makedirs(pred_dir, exist_ok=True)

    for fname in tqdm(os.listdir(gen_in_dir), desc=method_name):
        if not fname.endswith('.txt'):
            continue

        query   = read_file(os.path.join(gen_in_dir, fname))
        engine  = classifier_engine if use_classifier_retriever else rag_engine

        retrieval = engine.query(query)
        examples  = build_retrieval_dict(query, retrieval)
        prompt    = build_few_shot_prompt(query, examples)

        # Stage 1: classify
        category, system_text = classify_command(query)

        # Stage 2: generate
        raw_out = llama3(prompt, system_text)
        match   = re.search(r'br-t9s\.cli[^\n]*', raw_out)
        command = match.group(0).strip('\'"') if match else raw_out.strip()

        num = re.search(r'\d+', fname)
        if num:
            with open(os.path.join(pred_dir, f'response_in{num.group()}.txt'), 'w') as f:
                f.write(command)

    acc  = calculate_accuracy(gen_out_dir, pred_dir)
    miou = calculate_miou(gen_out_dir, pred_dir)
    print(f'{method_name}: Acc={acc:.2%}  MIoU={miou:.2%}')

In [ ]:
# Final results summary
print('\n' + '=' * 50)
print('  FINAL RESULTS SUMMARY')
print('=' * 50)
print('\nClassification Accuracy:')
print(f'  RAG — Classifier Retriever :  94.29%')
print(f'  RAG — Base Retriever       :  91.43%')
print(f'  Zero-Shot (no RAG)         :  74.29%')
print('\nGeneration (End-to-End Pipeline):')
print(f'  Exact match accuracy       :  ~45–46%')
print(f'  Mean IoU                   :  65–68%')
print('=' * 50)